<a href="https://colab.research.google.com/github/tiemtores/modele_IA_prev_AO/blob/main/TELECHARGEMENT_DONNEES_ERA_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TELECHARGEMENT DONNEES ERA5**

In [ ]:
pip install xarray netCDF4 requests cartopy geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import requests
import netCDF4 as nc

In [ ]:
a1,a2, m1, m2, j1, j2= (2026, 2027, 1, 3, 1, 32)

# **TELECHARGEMENT TEMPERATURE**

In [ ]:
#BONS CODE POUR LA CREATION DU FICHIER CDSPIRC POUR LE TELECHARGEMENT DES DONNEES TEMERATURE

# Étape 1 : installer cdsapi
!pip install cdsapi

# Étape 2 : configurer la clé API
import os
cds_key = "835b0994-fc88-451a-8e68-737dffdc97d0"  # ☢️ remplace par ta clé API exacte
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {cds_key}\n")

print("✅ Fichier .cdsapirc créé avec succès")

# Étape 3 : télécharger ERA5 par année et par mois
import cdsapi
c = cdsapi.Client()

for year in range(a1, a2):  # boucle de 2010 à 2011 inclus
    for month in range(m1, m2):  # boucle sur tous les mois
        month_str = f"{month:02d}"
        output_filename = f'/content/drive/MyDrive/Colab Notebooks/era5/temp/era5_temp_hourly_{year}_{month_str}.nc'
        print(f"Téléchargement pour l'année {year}, mois {month_str}...")
        c.retrieve(
            'reanalysis-era5-pressure-levels',
            {
                'product_type': 'reanalysis',
                'variable': 'temperature',
                'year': str(year),
                'month': month_str,  # Un seul mois par requête
                'day': [str(d).zfill(2) for d in range(j1, j2)],  # tous les jours
                'time': [f"{h:02d}:00" for h in range(24)],      # toutes les heures
                'pressure_level': ['200', '850'],
                'format': 'netcdf',
                'area': [18, -18, 4, 10],  # [N, W, S, E]
            },
            output_filename  # fichier par année et par mois
        )
        print(f"✅ Fichier {output_filename} téléchargé.")

print("Tous les fichiers horaires ont été téléchargés par année et par mois.")

✅ Fichier .cdsapirc créé avec succès
Téléchargement pour l'année 2026, mois 01...


2026-02-12 18:30:52,930 INFO Request ID is facc0436-dd8b-408a-9f79-39f4d0a07504
INFO:ecmwf.datastores.legacy_client:Request ID is facc0436-dd8b-408a-9f79-39f4d0a07504
2026-02-12 18:30:53,088 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-02-12 18:31:07,017 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


83e319fb4388cea5ba078f908620c9c0.nc:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/temp/era5_temp_hourly_2026_01.nc téléchargé.
Téléchargement pour l'année 2026, mois 02...


2026-02-12 18:31:09,984 INFO Request ID is 10128d80-6a02-43d8-b135-e393dc3c77b1
INFO:ecmwf.datastores.legacy_client:Request ID is 10128d80-6a02-43d8-b135-e393dc3c77b1
2026-02-12 18:31:10,298 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-02-12 18:31:19,146 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-02-12 18:32:00,830 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


ce6fceef003fc527f9ecfd94aee7f6dc.nc:   0%|          | 0.00/3.00M [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/temp/era5_temp_hourly_2026_02.nc téléchargé.
Tous les fichiers horaires ont été téléchargés par année et par mois.


In [ ]:
# BON CODE CONCATENATION ET RENOMMAGE DE CALIDTIME EN TIME POUR TEMPERATURE
import xarray as xr
import glob
import os
import pandas as pd

# Charger les fichiers horaires ERA5
data_path = "/content/drive/MyDrive/Colab Notebooks/era5/temp"
files = sorted(glob.glob(os.path.join(data_path, "*.nc")))

# Charger et concaténer
dst = xr.open_mfdataset(files, combine='by_coords', parallel=False)

# Vérifier les dimensions
print(dst.dims)

# ✅ Renommer la dimension 'valid_time' en 'time' si nécessaire
if 'valid_time' in dst.dims:
    dst = dst.rename({'valid_time': 'time'})

# 🕒 Convertir en datetime si besoin
dst['time'] = pd.to_datetime(dst['time'].values)

# ➕ Affecter un DateTimeIndex
dst = dst.assign_coords(time=dst['time'])

# ⚡ Agrégation journalière (moyenne des 24 heures)
ds_daily_temperature = dst['t'].resample(time="1D").mean()

# Sauvegarde du fichier final
output_file = os.path.join(data_path, "era5_temp_daily_mean_2026.nc")
if os.path.exists(output_file):
    os.remove(output_file)  # supprime l'ancien fichier si présent
ds_daily_temperature.to_netcdf(output_file)

print(f"✅ Données journalières sauvegardées dans {output_file}")

FrozenMappingWarningOnValuesAccess({'valid_time': 907, 'pressure_level': 2, 'latitude': 57, 'longitude': 113})
✅ Données journalières sauvegardées dans /content/drive/MyDrive/Colab Notebooks/era5/temp/era5_temp_daily_mean_2026.nc


# **DIVERGENCE**

In [ ]:
#BONS CODE POUR LA CREATION DU FICHIER CDSPIRC POUR LE TELECHARGEMENT DES DONNEES DIVERGENCE

# Étape 1 : installer cdsapi
!pip install cdsapi

# Étape 2 : configurer la clé API
import os
cds_key = "835b0994-fc88-451a-8e68-737dffdc97d0"  # ☢️ remplace par ta clé API exacte
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {cds_key}\n")

print("✅ Fichier .cdsapirc créé avec succès")

# Étape 3 : télécharger ERA5 par année et par mois
import cdsapi
c = cdsapi.Client()

for year in range(a1, a2):  # boucle de 2010 à 2011 inclus
    for month in range(m1, m2):  # boucle sur tous les mois
        month_str = f"{month:02d}"
        output_filename = f'/content/drive/MyDrive/Colab Notebooks/era5/divergence/era5_divergence_hourly_{year}_{month_str}.nc'
        print(f"Téléchargement pour l'année {year}, mois {month_str}...")
        c.retrieve(
            'reanalysis-era5-pressure-levels',
            {
                'product_type': 'reanalysis',
                'variable': 'divergence',
                'year': str(year),
                'month': month_str,  # Un seul mois par requête
                'day': [str(d).zfill(2) for d in range(j1, j2)],  # tous les jours
                'time': [f"{h:02d}:00" for h in range(24)],      # toutes les heures
                'pressure_level': ['200','600','700','850'],
                'format': 'netcdf',
                'area': [18, -18, 4, 10],  # [N, W, S, E]
            },
            output_filename  # fichier par année et par mois
        )
        print(f"✅ Fichier {output_filename} téléchargé.")

print("Tous les fichiers horaires ont été téléchargés par année et par mois.")

✅ Fichier .cdsapirc créé avec succès
Téléchargement pour l'année 2026, mois 01...


2026-02-12 18:34:52,419 INFO Request ID is 74134298-6b05-4d63-b3f4-58b3c4b324b6
INFO:ecmwf.datastores.legacy_client:Request ID is 74134298-6b05-4d63-b3f4-58b3c4b324b6
2026-02-12 18:34:52,578 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-02-12 18:35:14,291 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-02-12 18:35:25,840 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


a6c4d7a7c9059d1b3c951f38cb18ad4e.nc:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/divergence/era5_divergence_hourly_2026_01.nc téléchargé.
Téléchargement pour l'année 2026, mois 02...


2026-02-12 18:35:30,641 INFO Request ID is d43e3c77-e221-45db-8280-2b00059f60d9
INFO:ecmwf.datastores.legacy_client:Request ID is d43e3c77-e221-45db-8280-2b00059f60d9
2026-02-12 18:35:30,783 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-02-12 18:35:39,546 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-02-12 18:36:47,106 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


e140b121867d3e80b20c5eb195e3666.nc:   0%|          | 0.00/9.84M [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/divergence/era5_divergence_hourly_2026_02.nc téléchargé.
Tous les fichiers horaires ont été téléchargés par année et par mois.


In [ ]:
## BON CODE CONCATENATION ET RENOMMAGE DE CALIDTIME EN TIME POUR DIVERGENCE
import xarray as xr
import glob
import pandas as pd
import os

# Charger les fichiers horaires ERA5 divergence
data_path = "/content/drive/MyDrive/Colab Notebooks/era5/divergence"
files = sorted(glob.glob(os.path.join(data_path, "*.nc")))

# Charger et concaténer
dsd = xr.open_mfdataset(files, combine='by_coords', parallel=False)

# Vérifier les dimensions
print(dsd.dims)

# ✅ Renommer la dimension 'valid_time' en 'time' si nécessaire
if 'valid_time' in dsd.dims:
    dsd = dsd.rename({'valid_time': 'time'})

# 🕒 Convertir en datetime si besoin
dsd['time'] = pd.to_datetime(dsd['time'].values)

# ➕ Affecter un DateTimeIndex
dsd = dsd.assign_coords(time=dsd['time'])

# ⚡ Agrégation journalière (moyenne des 24 heures)
ds_daily_divergence = dsd['d'].resample(time="1D").mean()

# Sauvegarde du fichier final (corrigé)
output_file = "/content/drive/MyDrive/Colab Notebooks/era5/divergence/era5_divergence_daily_mean_2026.nc"
if os.path.exists(output_file):
    os.remove(output_file)  # supprime l'ancien fichier si présent
ds_daily_divergence.to_netcdf(output_file)

print(f"✅ Données journalières de divergence sauvegardées dans {output_file}")

FrozenMappingWarningOnValuesAccess({'valid_time': 907, 'pressure_level': 4, 'latitude': 57, 'longitude': 113})
✅ Données journalières de divergence sauvegardées dans /content/drive/MyDrive/Colab Notebooks/era5/divergence/era5_divergence_daily_mean_2026.nc


# **HUMIDITY**

7

In [ ]:
#BONS CODE POUR LA CREATION DU FICHIER CDSPIRC POUR LE TELECHARGEMENT DES DONNEES RELATIVE HUMIDITE

# Étape 1 : installer cdsapi
!pip install cdsapi

# Étape 2 : configurer la clé API
import os
cds_key = "835b0994-fc88-451a-8e68-737dffdc97d0"  # ☢️ remplace par ta clé API exacte
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {cds_key}\n")

print("✅ Fichier .cdsapirc créé avec succès")

# Étape 3 : télécharger ERA5 par année et par mois
import cdsapi
c = cdsapi.Client()

for year in range(a1, a2):  # boucle de 2010 à 2011 inclus
    for month in range(m1, m2):  # boucle sur tous les mois
        month_str = f"{month:02d}"
        output_filename = f'/content/drive/MyDrive/Colab Notebooks/era5/rh/era5_rh_hourly_{year}_{month_str}.nc'
        print(f"Téléchargement pour l'année {year}, mois {month_str}...")
        c.retrieve(
            'reanalysis-era5-pressure-levels',
            {
                'product_type': 'reanalysis',
                'variable': 'relative_humidity',
                'year': str(year),
                'month': month_str,  # Un seul mois par requête
                'day': [str(d).zfill(2) for d in range(j1, j2)],  # tous les jours
                'time': [f"{h:02d}:00" for h in range(24)],      # toutes les heures
                'pressure_level': ['200','600','700','850'],
                'format': 'netcdf',
                'area': [18, -18, 4, 10],  # [N, W, S, E]
            },
            output_filename  # fichier par année et par mois
        )
        print(f"✅ Fichier {output_filename} téléchargé.")

print("Tous les fichiers horaires ont été téléchargés par année et par mois.")

✅ Fichier .cdsapirc créé avec succès
Téléchargement pour l'année 2026, mois 01...


2026-02-12 17:56:26,445 INFO Request ID is f357935d-f40b-4673-9c39-c02bb6010452
INFO:ecmwf.datastores.legacy_client:Request ID is f357935d-f40b-4673-9c39-c02bb6010452
2026-02-12 17:56:26,604 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-02-12 17:57:17,129 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-02-12 18:02:49,150 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


968db07cc3109ba899779dde8b20c494.nc:   0%|          | 0.00/37.5M [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/rh/era5_rh_hourly_2026_01.nc téléchargé.
Téléchargement pour l'année 2026, mois 02...


2026-02-12 18:05:35,411 INFO Request ID is 03ac9046-22c9-4d3d-afcb-cc3f189792ab
INFO:ecmwf.datastores.legacy_client:Request ID is 03ac9046-22c9-4d3d-afcb-cc3f189792ab
2026-02-12 18:05:35,588 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-02-12 18:05:44,472 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-02-12 18:06:52,270 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


be360f8bc986d9ee5d45243a6122fdc3.nc:   0%|          | 0.00/8.35M [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/rh/era5_rh_hourly_2026_02.nc téléchargé.
Tous les fichiers horaires ont été téléchargés par année et par mois.


In [ ]:
import xarray as xr
import glob
import pandas as pd
import os

# 📂 Chemin vers les fichiers horaires ERA5 humidité relative
data_path = "/content/drive/MyDrive/Colab Notebooks/era5/rh"
files = sorted(glob.glob(os.path.join(data_path, "*.nc")))

# Charger et concaténer
dsr = xr.open_mfdataset(files, combine='by_coords', parallel=False)

# Vérifier les dimensions
print(dsr.dims)

# ✅ Renommer la dimension 'valid_time' en 'time' si nécessaire
if 'valid_time' in dsr.dims:
    dsr = dsr.rename({'valid_time': 'time'})

# 🕒 Convertir en datetime si besoin
dsr['time'] = pd.to_datetime(dsr['time'].values)

# ➕ Affecter un DateTimeIndex
dsr = dsr.assign_coords(time=dsr['time'])

# ⚡ Agrégation journalière (moyenne des 24 heures)
ds_daily_rh = dsr['r'].resample(time="1D").mean()

# Sauvegarde du fichier final
output_file = "/content/drive/MyDrive/Colab Notebooks/era5/rh/era5_rh_daily_mean_2026.nc"
if os.path.exists(output_file):
    os.remove(output_file)  # supprime l'ancien fichier si présent
ds_daily_rh.to_netcdf(output_file)

print(f"✅ Données journalières d'humidité relative sauvegardées dans {output_file}")

FrozenMappingWarningOnValuesAccess({'valid_time': 907, 'pressure_level': 4, 'latitude': 57, 'longitude': 113})
✅ Données journalières d'humidité relative sauvegardées dans /content/drive/MyDrive/Colab Notebooks/era5/rh/era5_rh_daily_mean_2026.nc


# **TELECHARGEMENT CAPE**

In [ ]:
# Étape 1 : installer cdsapi
!pip install cdsapi

# Étape 2 : configurer la clé API
import os
cds_key = "835b0994-fc88-451a-8e68-737dffdc97d0"  # ⚠️ remplace par ta clé API exacte
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {cds_key}\n")

print("✅ Fichier .cdsapirc créé avec succès")

# Étape 3 : télécharger ERA5 CAPE par année et par mois
import cdsapi
c = cdsapi.Client()

for year in range(a1, a2):  # boucle de 2010 à 2020 inclus
    for month in range(m1, m2):  # boucle sur tous les mois
        month_str = f"{month:02d}"
        output_filename = f'/content/drive/MyDrive/Colab Notebooks/era5/cape/era5_cape_hourly_{year}_{month_str}.nc'
        print(f"Téléchargement pour l'année {year}, mois {month_str}...")
        c.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type': 'reanalysis',
                'variable': 'cape',
                'year': str(year),
                'month': month_str,  # Un seul mois par requête
                'day': [str(d).zfill(2) for d in range(j1, j2)],  # tous les jours
                'time': [f"{h:02d}:00" for h in range(24)],      # toutes les heures
                'format': 'netcdf',
                'area': [18, -18, 4, 10],  # [N, W, S, E] → exemple Afrique de l’Ouest
            },
            output_filename  # fichier par année et par mois
        )
        print(f"✅ Fichier {output_filename} téléchargé.")

print("Tous les fichiers horaires CAPE ont été téléchargés par année et par mois.")

✅ Fichier .cdsapirc créé avec succès
Téléchargement pour l'année 2026, mois 01...


2026-02-12 18:15:17,955 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

2df1a80ab78fa65b049a513916882fce.nc:   0%|          | 0.00/3.32M [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/cape/era5_cape_hourly_2026_01.nc téléchargé.
Téléchargement pour l'année 2026, mois 02...


2026-02-12 18:17:16,197 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

4c023aa5a09f3785e49ae64478df6e0b.nc:   0%|          | 0.00/900k [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/cape/era5_cape_hourly_2026_02.nc téléchargé.
Tous les fichiers horaires CAPE ont été téléchargés par année et par mois.


In [ ]:
import xarray as xr
import glob
import pandas as pd
import os

# 📂 Chemin vers les fichiers horaires ERA5 CAPE
data_path = "/content/drive/MyDrive/Colab Notebooks/era5/cape"
files = sorted(glob.glob(os.path.join(data_path, "*.nc")))

# Charger et concaténer
dsc = xr.open_mfdataset(files, combine='by_coords', parallel=False)

# Vérifier les dimensions
print(dsc.dims)

# ✅ Renommer la dimension 'valid_time' en 'time' si nécessaire
if 'valid_time' in dsc.dims:
    dsc = dsc.rename({'valid_time': 'time'})

# 🕒 Convertir en datetime si besoin
dsc['time'] = pd.to_datetime(dsc['time'].values)

# ➕ Affecter un DateTimeIndex
dsc = dsc.assign_coords(time=dsc['time'])

# ⚡ Agrégation journalière (moyenne des 24 heures)
ds_daily_cape = dsc['cape'].resample(time="1D").mean()

# Sauvegarde du fichier final
output_file = "/content/drive/MyDrive/Colab Notebooks/era5/cape/era5_cape_daily_mean_20126.nc"
if os.path.exists(output_file):
    os.remove(output_file)  # supprime l'ancien fichier si présent
ds_daily_cape.to_netcdf(output_file)

print(f"✅ Données journalières CAPE sauvegardées dans {output_file}")

FrozenMappingWarningOnValuesAccess({'valid_time': 907, 'latitude': 57, 'longitude': 113})
✅ Données journalières CAPE sauvegardées dans /content/drive/MyDrive/Colab Notebooks/era5/cape/era5_cape_daily_mean_20126.nc


# **TELECHARGEMENT DONNEES PRECIPITABLE WATER**

In [ ]:
# Étape 1 : installer cdsapi
!pip install cdsapi

# Étape 2 : configurer la clé API
import os
cds_key = "835b0994-fc88-451a-8e68-737dffdc97d0"  # ⚠️ remplace par ta clé API exacte
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {cds_key}\n")

print("✅ Fichier .cdsapirc créé avec succès")

# Étape 3 : télécharger ERA5 TCRW par année et par mois
import cdsapi
c = cdsapi.Client()

for year in range(a1, a2):  # boucle de 2010 à 2020 inclus
    for month in range(m1, m2):  # boucle sur tous les mois
        month_str = f"{month:02d}"
        output_filename = f'/content/drive/MyDrive/Colab Notebooks/era5/tcrw/era5_tcrw_hourly_{year}_{month_str}.nc'
        print(f"Téléchargement pour l'année {year}, mois {month_str}...")
        c.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type': 'reanalysis',
                'variable': 'total_column_rain_water',  # TCRW
                'year': str(year),
                'month': month_str,  # Un seul mois par requête
                'day': [str(d).zfill(2) for d in range(j1, j2)],  # tous les jours
                'time': [f"{h:02d}:00" for h in range(24)],      # toutes les heures
                'format': 'netcdf',
                'area': [18, -18, 4, 10],  # [N, W, S, E] → exemple Afrique de l’Ouest
            },
            output_filename  # fichier par année et par mois
        )
        print(f"✅ Fichier {output_filename} téléchargé.")

print("Tous les fichiers horaires TCRW ont été téléchargés par année et par mois.")

✅ Fichier .cdsapirc créé avec succès
Téléchargement pour l'année 2026, mois 01...


2026-02-12 18:19:19,504 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

56456e7d36a1129fc54a2a5601ab54ed.nc:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/tcrw/era5_tcrw_hourly_2026_01.nc téléchargé.
Téléchargement pour l'année 2026, mois 02...


2026-02-12 18:21:17,336 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

dec2455aa8ab3a7f9b33eec342c0a1c8.nc:   0%|          | 0.00/307k [00:00<?, ?B/s]

✅ Fichier /content/drive/MyDrive/Colab Notebooks/era5/tcrw/era5_tcrw_hourly_2026_02.nc téléchargé.
Tous les fichiers horaires TCRW ont été téléchargés par année et par mois.


In [ ]:
import xarray as xr
import glob
import pandas as pd
import os

# 📂 Chemin vers les fichiers horaires ERA5 TCRW
data_path = "/content/drive/MyDrive/Colab Notebooks/era5/tcrw"
files = sorted(glob.glob(os.path.join(data_path, "*.nc")))

# Charger et concaténer
dsw = xr.open_mfdataset(files, combine='by_coords', parallel=False)

# Vérifier les dimensions
print(dsw.dims)

# ✅ Renommer la dimension 'valid_time' en 'time' si nécessaire
if 'valid_time' in dsw.dims:
    dsw = dsw.rename({'valid_time': 'time'})

# 🕒 Convertir en datetime si besoin
dsw['time'] = pd.to_datetime(dsw['time'].values)

# ➕ Affecter un DateTimeIndex
dsw = dsw.assign_coords(time=dsw['time'])

# ⚡ Agrégation journalière (moyenne des 24 heures)
ds_daily_tcrw = dsw['tcrw'].resample(time="1D").sum()

# Sauvegarde du fichier final
output_file = "/content/drive/MyDrive/Colab Notebooks/era5/tcrw/era5_tcrw_daily_mean_2026.nc"
if os.path.exists(output_file):
    os.remove(output_file)  # supprime l'ancien fichier si présent
ds_daily_tcrw.to_netcdf(output_file)

print(f"✅ Données journalières TCRW sauvegardées dans {output_file}")

FrozenMappingWarningOnValuesAccess({'valid_time': 907, 'latitude': 57, 'longitude': 113})
✅ Données journalières TCRW sauvegardées dans /content/drive/MyDrive/Colab Notebooks/era5/tcrw/era5_tcrw_daily_mean_2026.nc
